In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os
import glob
import numpy as np
!pip install SimpleITK
import SimpleITK as sitk


def preprocess_subject(image_path, mask_path, output_dir, target_size=(64, 64, 64)):
    """
    Preprocesses a single MSD Hippocampus subject:
    1. Resamples image (Linear) and mask (Nearest Neighbor) to 1.0mm isotropic.
    2. Resizes/Pads both to a uniform target_size for deep learning CNNs.
    3. Standardizes image intensities using foreground-only Z-score normalization.
    4. Verifies the 3-way {0, 1, 2} labels are preserved.
    """
    subject_id = os.path.basename(image_path).split('.')[0]
    # Remove '(1)' if it exists due to download issues
    if '(' in subject_id and ')' in subject_id:
        subject_id = subject_id.split('(')[0]

    # 1. Load Original Volumes
    src_img = sitk.ReadImage(image_path)
    src_msk = sitk.ReadImage(mask_path)

    # 2. Resample to Isotropic 1.0mm x 1.0mm x 1.0mm
    target_spacing = (1.0, 1.0, 1.0)

    # Compute new size based on original physical size and target spacing
    original_size = src_img.GetSize()
    original_spacing = src_img.GetSpacing()
    new_size = [
        int(round(original_size[i] * original_spacing[i] / target_spacing[i]))
        for i in range(3)
    ]

    # Resample Image using Linear Interpolation
    resample_img = sitk.ResampleImageFilter()
    resample_img.SetOutputSpacing(target_spacing)
    resample_img.SetSize(new_size)
    resample_img.SetOutputDirection(src_img.GetDirection())
    resample_img.SetOutputOrigin(src_img.GetOrigin())
    resample_img.SetInterpolator(sitk.sitkLinear)
    res_img = resample_img.Execute(src_img)

    # Resample Mask using Nearest Neighbor Interpolation (CRITICAL for {0,1,2})
    resample_msk = sitk.ResampleImageFilter()
    resample_msk.SetOutputSpacing(target_spacing)
    resample_msk.SetSize(new_size)
    resample_msk.SetOutputDirection(src_msk.GetDirection())
    resample_msk.SetOutputOrigin(src_msk.GetOrigin())
    resample_msk.SetInterpolator(sitk.sitkNearestNeighbor) # Keeps labels discrete
    res_msk = resample_msk.Execute(src_msk)

    # 3. Standardize Size (Pad/Crop to target_size for CNN batching)
    def pad_crop_to_target(sitk_vol):
        current_size = sitk_vol.GetSize() # SimpleITK uses (X, Y, Z)

        pad_low = [max(0, (target_size[i] - current_size[i]) // 2) for i in range(3)]
        pad_high = [max(0, target_size[i] - current_size[i] - pad_low[i]) for i in range(3)]

        # Pad if smaller than target
        if sum(pad_low) + sum(pad_high) > 0:
            sitk_vol = sitk.ConstantPad(sitk_vol, pad_low, pad_high, 0.0)

        # Recalculate size after padding to check for cropping requirements
        current_size = sitk_vol.GetSize()
        if current_size != target_size:
            crop_low = [max(0, (current_size[i] - target_size[i]) // 2) for i in range(3)]
            crop_high = [max(0, current_size[i] - target_size[i] - crop_low[i]) for i in range(3)]
            sitk_vol = sitk.Crop(sitk_vol, crop_low, crop_high)

        return sitk_vol

    final_img = pad_crop_to_target(res_img)
    final_msk = pad_crop_to_target(res_msk)

    # Convert to numpy arrays (SimpleITK to Numpy changes ordering to [Z, Y, X])
    img_arr = sitk.GetArrayFromImage(final_img)
    msk_arr = sitk.GetArrayFromImage(final_msk).astype(np.int16)

    # 4. Foreground Z-Score Intensity Normalization
    foreground_mask = img_arr > 0
    if np.any(foreground_mask):
        mean = np.mean(img_arr[foreground_mask])
        std = np.std(img_arr[foreground_mask])
        normalized_img_arr = (img_arr - mean) / (std + 1e-8)
        normalized_img_arr[~foreground_mask] = 0
    else:
        normalized_img_arr = img_arr

    # 5. Safety Check: Verify unique labels
    unique_labels = np.unique(msk_arr)

    # Convert back to SimpleITK structures
    out_img = sitk.GetImageFromArray(normalized_img_arr)
    out_img.CopyInformation(final_img)

    out_msk = sitk.GetImageFromArray(msk_arr)
    out_msk.CopyInformation(final_msk)

    # Save Outputs
    os.makedirs(os.path.join(output_dir, "imagesTr"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labelsTr"), exist_ok=True)

    sitk.WriteImage(out_img, os.path.join(output_dir, "imagesTr", f"{subject_id}.nii.gz"))
    sitk.WriteImage(out_msk, os.path.join(output_dir, "labelsTr", f"{subject_id}.nii.gz"))

    return unique_labels

# --- Execution Block ---
if __name__ == "__main__":
    ORIGINAL_IMG_DIR = "/content/drive/MyDrive/Decathlon Hippocampus/imagesTr"
    ORIGINAL_MSK_DIR = "/content/drive/MyDrive/Decathlon Hippocampus/labelsTr"
    OUTPUT_DIR = "/content/drive/MyDrive/preprocessed_Decathlon_Anterior_Posterior_32_32_32"

    TARGET_SHAPE = (32,32,32)

    # Modified glob to explicitly look for .nii and .nii.gz files
    images = sorted(glob.glob(os.path.join(ORIGINAL_IMG_DIR, "*.nii")) + glob.glob(os.path.join(ORIGINAL_IMG_DIR, "*.nii.gz")))
    masks = sorted(glob.glob(os.path.join(ORIGINAL_MSK_DIR, "*.nii")) + glob.glob(os.path.join(ORIGINAL_MSK_DIR, "*.nii.gz")))

    print(f"🚀 Processing {len(images)} subjects for real 3-way task...")
    for img_p, msk_p in zip(images, masks):
        labels = preprocess_subject(img_p, msk_p, OUTPUT_DIR, target_size=TARGET_SHAPE)
        print(f"Processed {os.path.basename(img_p)} -> Preserved Labels: {labels}")
    print("\n✅ Done! Dataset ready for deep learning multiclass segmentation.")